# IGF1R-3QQU - Calx-β  Molecular Dynamics with Gromacs

Load and interactively visualize files in `../data/3d_struct/` with **nglview**, then (optional) run a **GROMACS** binding-site MD simulation of **calx-β** on **IGF1R (3QQU)**.  

**Input Files:**  
| File | Contents |
| --- | --- |
| 3QQU.pdbqs | IGF1R kinase receptor (AutoDock grid map source) |
| calxbeta_18.39.pdb | Calx-β docked to IGF1R (−18.39 kcal/mol) |
  


  


**Not used here yet:**  
| File | Contents |
| --- | --- |
| `IVM_3qqu_mid_17.25.pdb` | Ivermectin docked pose, run 13 (−17.25 kcal/mol) |
| `IVM_17.45_3QQU.pdb` | Ivermectin docked pose, run 11 (−17.45 kcal/mol) |

**Run in the MD container** (`./up-md.sh --build` from `tomo_2026b/md/`).

In [ ]:
from __future__ import annotations

import io
import json
import math
import re
import shutil
import subprocess
import tempfile
import time
from collections import defaultdict
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import mdtraj as md
import nglview as ng
import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display

# Optional at import time (solvation / PDB repair). Fail later if missing when used.
try:
    from openmm.app import PDBFile
    from pdbfixer import PDBFixer
except ImportError:
    PDBFile = None  # type: ignore[misc, assignment]
    PDBFixer = None  # type: ignore[misc, assignment]

# --- Paths ---
STRUCT_DIR = next(
    p.resolve()
    for p in (
        Path("../data/3d_struct"),
        Path("/workspace/data/3d_struct"),
        Path("data/3d_struct"),
    )
    if p.is_dir()
)
STRUCT_FILES = sorted(
    f for f in STRUCT_DIR.iterdir() if f.is_file() and not f.name.startswith(".")
)
RECEPTOR_PATH = STRUCT_DIR / "3QQU.pdbqs"
LIGAND_PATH = STRUCT_DIR / "calxbeta_18.39.pdb"
MD_ROOT = STRUCT_DIR.parent / "md_calx_3qqu"
MD_ROOT.mkdir(parents=True, exist_ok=True)
RUNS_DIR = STRUCT_DIR.parent / "runs"
RUNS_DIR.mkdir(parents=True, exist_ok=True)
MD_TMP_ROOT = Path("/tmp/md_calx_3qqu")  # local disk for production (not gcsfuse)

# --- Display ---
CALX_BETA_CARTOON_COLOR = "#377eb8"
RECEPTOR_POCKET_CARTOON_COLOR = "#e41a1c"

# --- MD pipeline ---
RUN_MD = True  # False = build inputs only (no gmx)
FORCE_FROM_SCRATCH = False  # True = ignore md.cpt and re-run full pipeline
FORCE_MD_REPROCESS = True  # regenerate md_final_protein / energy from trajectory
FORCE_BINDING_REPROCESS = True  # recompute RMSD / contact metrics

# 100 ns production: 50_000_000 steps × 0.002 ps = 100_000 ps.
TOTAL_STEPS = 50_000_000
MD_DT_PS = 0.002  # timestep in ps (0.002 ps = 2 fs)
TRAJ_SAVE_PS = 10.0  # xtc / energy dump interval (ps)
CHECKPOINT_SAVE_PS = 1000.0  # checkpoint interval (ps); 1 ns
CHECKPOINT_SAVE_STEPS = max(1, int(round(CHECKPOINT_SAVE_PS / MD_DT_PS)))  # 500000

USE_GPU_MDRUN = True  # requires CUDA gmx (./md-run.sh --gpu)
MDRUN_GPU_FLAGS = ["-nb", "gpu", "-pme", "gpu", "-bonded", "gpu", "-update", "gpu"]

EM_STEPS = min(5_000, max(500, TOTAL_STEPS))
NVT_STEPS = min(25_000, max(500, TOTAL_STEPS))
NPT_STEPS = min(25_000, max(500, TOTAL_STEPS))
POCKET_CUTOFF_NM = 1.0
FORCEFIELD = "amber99sb-ildn"
WATER_MODEL = "tip3p"


def _fmt_config_value(value) -> str:
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, (list, tuple)):
        return " ".join(str(v) for v in value)
    if isinstance(value, float):
        return f"{value:g}"
    return str(value)


def show_config_table() -> None:
    """Tabular view of notebook configuration (name, value, purpose)."""
    rows = [
        ("STRUCT_DIR", STRUCT_DIR, "Folder with input receptor / ligand structures"),
        ("STRUCT_FILES", f"{len(STRUCT_FILES)} files", "List of structure files found under STRUCT_DIR"),
        ("RECEPTOR_PATH", RECEPTOR_PATH, "IGF1R kinase structure (3QQU)"),
        ("LIGAND_PATH", LIGAND_PATH, "Docked calx-β pose used to build the complex"),
        ("MD_ROOT", MD_ROOT, "Workspace MD directory (topology, equilibration, synced outputs)"),
        ("RUNS_DIR", RUNS_DIR, "Archive directory for copied run logs"),
        ("MD_TMP_ROOT", MD_TMP_ROOT, "Local /tmp path for production mdrun (avoids gcsfuse)"),
        ("CALX_BETA_CARTOON_COLOR", CALX_BETA_CARTOON_COLOR, "nglview / PNG color for calx-β"),
        ("RECEPTOR_POCKET_CARTOON_COLOR", RECEPTOR_POCKET_CARTOON_COLOR, "nglview / PNG color for receptor pocket"),
        ("RUN_MD", RUN_MD, "If True, run GROMACS; if False, only build pocket / inputs"),
        ("FORCE_FROM_SCRATCH", FORCE_FROM_SCRATCH, "If True, ignore md.cpt and re-run solvation + equilibration"),
        ("FORCE_MD_REPROCESS", FORCE_MD_REPROCESS, "If True, rebuild final PDB / energy plots from the trajectory"),
        ("FORCE_BINDING_REPROCESS", FORCE_BINDING_REPROCESS, "If True, recompute ligand RMSD and contact metrics"),
        ("TOTAL_STEPS", TOTAL_STEPS, "Production MD step count (simulation length = TOTAL_STEPS × MD_DT_PS)"),
        ("MD_DT_PS", MD_DT_PS, "Integration timestep in picoseconds (0.002 ps = 2 fs)"),
        ("production_ps", TOTAL_STEPS * MD_DT_PS, "Derived production length in picoseconds"),
        ("TRAJ_SAVE_PS", TRAJ_SAVE_PS, "Interval for writing trajectory / energy frames (ps)"),
        ("CHECKPOINT_SAVE_PS", CHECKPOINT_SAVE_PS, "Checkpoint write interval in picoseconds (1000 ps = 1 ns)"),
        ("CHECKPOINT_SAVE_STEPS", CHECKPOINT_SAVE_STEPS, "Checkpoint interval in steps (from CHECKPOINT_SAVE_PS / MD_DT_PS)"),
        ("USE_GPU_MDRUN", USE_GPU_MDRUN, "If True, offload production mdrun nonbonded/PME/bonded/update to GPU"),
        ("MDRUN_GPU_FLAGS", MDRUN_GPU_FLAGS, "gmx mdrun GPU offload flags used when USE_GPU_MDRUN is True"),
        ("EM_STEPS", EM_STEPS, "Energy-minimization step count"),
        ("NVT_STEPS", NVT_STEPS, "NVT equilibration step count"),
        ("NPT_STEPS", NPT_STEPS, "NPT equilibration step count"),
        ("POCKET_CUTOFF_NM", POCKET_CUTOFF_NM, "Receptor atoms kept near ligand when building the pocket (nm)"),
        ("FORCEFIELD", FORCEFIELD, "GROMACS protein force field for pdb2gmx"),
        ("WATER_MODEL", WATER_MODEL, "Water model paired with the force field"),
    ]
    config_df = pd.DataFrame(
        [
            {"name": name, "value": _fmt_config_value(value), "what_it_does": desc}
            for name, value, desc in rows
        ]
    )
    styled = (
        config_df.style.hide(axis="index")
        .set_properties(
            subset=["name"],
            **{"text-align": "left", "white-space": "nowrap"},
        )
        .set_properties(
            subset=["what_it_does"],
            **{
                "text-align": "left",
                "white-space": "normal",
                "overflow-wrap": "anywhere",
                "max-width": "28rem",
            },
        )
        .set_table_styles(
            [
                {"selector": "th", "props": [("text-align", "left")]},
                {"selector": "td", "props": [("vertical-align", "top")]},
            ],
            overwrite=False,
        )
    )
    display(styled)
    return None


print(f"Structure folder: {STRUCT_DIR}")
print(f"Found {len(STRUCT_FILES)} file(s)")
print(
    f"MD_ROOT={MD_ROOT}  production={TOTAL_STEPS * MD_DT_PS:.0f} ps  "
    f"RUN_MD={RUN_MD}  USE_GPU_MDRUN={USE_GPU_MDRUN}"
)


In [ ]:
# Re-display configuration. Requires the previous cell (imports + config) to have been run.
if "show_config_table" not in globals():
    raise NameError(
        "Configuration not loaded. Run the previous cell (imports + config) first, "
        "then re-run this cell."
    )
show_config_table()


In [ ]:
_PDB_MARKERS = ("ATOM", "HETATM", "MODEL", "ENDMDL", "TER", "USER")


def _normalize_pdb_line(line: str) -> str | None:
    line = line.strip().rstrip("\\")
    if not any(marker in line for marker in _PDB_MARKERS):
        return None
    for marker in _PDB_MARKERS:
        if marker in line:
            line = line[line.find(marker) :]
            break
    return line if line.startswith(_PDB_MARKERS) else None


def read_structure_text(path: Path) -> tuple[str, list[str]]:
    """Return PDB-like text and any load warnings."""
    raw = path.read_text(errors="replace")
    warnings: list[str] = []

    if raw.lstrip().startswith("{\\rtf"):
        warnings.append("RTF wrapper detected — extracting ATOM/MODEL records")
        cleaned: list[str] = []
        for line in raw.replace("\\\n", "\n").splitlines():
            norm = _normalize_pdb_line(line)
            if norm:
                cleaned.append(norm)
        raw = "\n".join(cleaned) + "\n"

    if path.suffix.lower() == ".pdbqs":
        warnings.append("Truncating .pdbqs ATOM records to 80-column PDB for nglview")
        lines = []
        for line in raw.splitlines():
            if line.startswith(("ATOM", "HETATM")):
                lines.append(line[:80])
        raw = "\n".join(lines) + "\n"

    return raw, warnings


def count_residues(text: str) -> int:
    """Unique residues from ATOM/HETATM lines: (chain, resSeq, insertion code)."""
    residues: set[tuple[str, str, str]] = set()
    for line in text.splitlines():
        if not line.startswith(("ATOM", "HETATM")) or len(line) < 26:
            continue
        chain = line[21] if len(line) > 21 else " "
        resseq = line[22:26].strip()
        icode = line[26] if len(line) > 26 else " "
        if resseq:
            residues.add((chain, resseq, icode))
    return len(residues)


def structure_summary(path: Path) -> dict:
    text, warnings = read_structure_text(path)
    n_atom = sum(1 for line in text.splitlines() if line.startswith(("ATOM", "HETATM")))
    n_model = len(re.findall(r"^MODEL", text, flags=re.MULTILINE)) or (1 if n_atom else 0)
    user_energy = re.search(r"Final Docked Energy\s*=\s*([-\d.]+)", text)
    return {
        "file": path.name,
        "suffix": path.suffix,
        "n_atoms": n_atom,
        "n_residues": count_residues(text),
        "n_models": n_model,
        "docked_energy_kcal_mol": float(user_energy.group(1)) if user_energy else None,
        "warnings": "; ".join(warnings) if warnings else "",
    }


summary_df = pd.DataFrame(structure_summary(p) for p in STRUCT_FILES)
summary_df

In [ ]:
def make_view(path: Path, *, width: int = 600, height: int = 450):
    """Build an nglview widget for one structure file."""
    text, warnings = read_structure_text(path)
    if not text.strip():
        raise ValueError(f"No coordinates parsed from {path.name}")

    suffix = ".pdb" if path.suffix.lower() in {".pdb", ".pdbqs"} else path.suffix
    with tempfile.NamedTemporaryFile(mode="w", suffix=suffix, delete=False) as handle:
        handle.write(text)
        tmp_path = handle.name

    view = ng.show_file(tmp_path, default=True)
    view.clear_representations()

    meta = structure_summary(path)
    if meta["n_atoms"] <= 120:
        view.add_representation("ball+stick", color_scheme="element")
    else:
        view.add_representation("cartoon", color_scheme="chainindex")
        view.add_representation("licorice", sele="hetero and not (water or ion)", color_scheme="element")

    view.center()
    view._tomo_warnings = warnings  # noqa: SLF001 — notebook-only metadata
    return view


def _pdb_ca_coords_by_residue(
    pdb_path: Path,
    *,
    ligand_chain: str = "B",
    receptor_only: bool = False,
    chain_id: str | None = None,
) -> dict[tuple[str, int], np.ndarray]:
    """CA coordinates keyed by (resname, resSeq)."""
    coords: dict[tuple[str, int], np.ndarray] = {}
    for line in pdb_path.read_text().splitlines():
        if not line.startswith("ATOM") or line[12:16].strip() != "CA":
            continue
        chain = line[21]
        if chain_id is not None and chain != chain_id:
            continue
        if receptor_only and chain == ligand_chain:
            continue
        key = (line[17:20].strip(), int(line[22:26].strip()))
        coords[key] = np.array(
            [float(line[30:38]), float(line[38:46]), float(line[46:54])],
            dtype=float,
        )
    return coords


def _kabsch_fit(mobile: np.ndarray, target: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Return R, t such that mobile @ R + t ≈ target (row-vector convention)."""
    mobile_c = mobile - mobile.mean(axis=0)
    target_c = target - target.mean(axis=0)
    H = mobile_c.T @ target_c
    U, _, Vt = np.linalg.svd(H)
    R = Vt.T @ U.T
    if np.linalg.det(R) < 0:
        Vt[-1, :] *= -1
        R = Vt.T @ U.T
    t = target.mean(axis=0) - mobile.mean(axis=0) @ R
    return R, t


def _transform_pdb_atom_lines(text: str, R: np.ndarray, t: np.ndarray) -> str:
    """Apply a rigid transform to ATOM/HETATM coordinates in PDB text."""
    lines: list[str] = []
    for line in text.splitlines():
        if not line.startswith(("ATOM", "HETATM")) or len(line) < 54:
            continue
        x, y, z = float(line[30:38]), float(line[38:46]), float(line[46:54])
        new = np.array([x, y, z], dtype=float) @ R + t
        base = line.ljust(54)
        lines.append(f"{base[:30]}{new[0]:8.3f}{new[1]:8.3f}{new[2]:8.3f}{base[46:]}")
    return "\n".join(lines) + "\nTER\nEND\n"


def _ngl_pocket_residue_selection(pocket_pdb: Path, *, ligand_chain: str = "B") -> str:
    """NGL selection string for receptor pocket residues (non-ligand chains)."""
    resseqs = sorted(
        {
            int(line[22:26].strip())
            for line in pocket_pdb.read_text().splitlines()
            if line.startswith("ATOM") and line[21] != ligand_chain
        }
    )
    if not resseqs:
        return "none"
    return " or ".join(str(r) for r in resseqs)


def _write_temp_pdb(text: str) -> str:
    with tempfile.NamedTemporaryFile(mode="w", suffix=".pdb", delete=False) as handle:
        handle.write(text.strip() + "\n")
        return handle.name


def _md_final_ligand_pdb_text(pdb_path: Path, *, ligand_chain: str = "B") -> str:
    """Extract calx-β chain from the MD final pocket PDB."""
    lines = [
        line
        for line in pdb_path.read_text().splitlines()
        if line.startswith(("ATOM", "HETATM")) and line[21] == ligand_chain
    ]
    if not lines:
        raise ValueError(f"No chain {ligand_chain} atoms in {pdb_path.name}")
    return "\n".join(lines) + "\nTER\nEND\n"


def make_md_final_view(
    pdb_path: Path,
    *,
    receptor_path: Path | None = None,
    pocket_pdb: Path | None = None,
    full_receptor_opacity: float = 0.35,
):
    """Full IGF1R context (gray) + pocket highlight (red) + calx-β (blue)."""
    text, _ = read_structure_text(pdb_path)
    if not text.strip():
        raise ValueError(f"No coordinates parsed from {pdb_path.name}")

    if receptor_path is None or not receptor_path.exists():
        view = ng.show_file(_write_temp_pdb(text), default=False)
        view.add_representation(
            "cartoon",
            color=RECEPTOR_POCKET_CARTOON_COLOR,
            sele="not :B",
        )
        view.add_representation(
            "cartoon",
            color=CALX_BETA_CARTOON_COLOR,
            sele=":B",
        )
        view.center()
        return view

    pocket_pdb = pocket_pdb if pocket_pdb is not None and pocket_pdb.exists() else pdb_path
    ref_ca = _pdb_ca_coords_by_residue(pdb_path, receptor_only=True)
    mob_ca = _pdb_ca_coords_by_residue(receptor_path, chain_id="A")
    shared = sorted(set(ref_ca) & set(mob_ca))

    receptor_text, _ = read_structure_text(receptor_path)
    if len(shared) >= 3:
        ref_xyz = np.stack([ref_ca[k] for k in shared])
        mob_xyz = np.stack([mob_ca[k] for k in shared])
        R, t = _kabsch_fit(mob_xyz, ref_xyz)
        receptor_text = _transform_pdb_atom_lines(receptor_text, R, t)
    else:
        print(
            f"Warning: only {len(shared)} shared pocket CAs for alignment — "
            "using static receptor coordinates.",
            flush=True,
        )

    pocket_sele = _ngl_pocket_residue_selection(pocket_pdb)
    view = ng.show_file(_write_temp_pdb(receptor_text), default=False)
    view.add_representation(
        "cartoon",
        color="#bdbdbd",
        opacity=full_receptor_opacity,
    )
    if pocket_sele != "none":
        view.add_representation("cartoon", color="red", sele=pocket_sele)

    ligand_text = _md_final_ligand_pdb_text(pdb_path)
    # add_component defaults to sstruc cartoon (yellow/green beta sheets) — disable it.
    view.add_component(_write_temp_pdb(ligand_text), default_representation=False)
    view.add_representation(
        "cartoon",
        color=CALX_BETA_CARTOON_COLOR,
        component=1,
    )
    view.center()
    return view


def _is_hydrogen(atom_name: str) -> bool:
    name = atom_name.strip()
    return name.startswith(("H", "D"))


def _pdb_ca_traces(
    pdb_path: Path,
    *,
    chain_id: str | None = None,
    max_segment_angstrom: float = 6.0,
) -> list[list[tuple[float, float, float]]]:
    """CA coordinates for one chain, sorted by residue number and split at large gaps."""
    entries: list[tuple[int, tuple[float, float, float]]] = []
    for line in pdb_path.read_text().splitlines():
        if not line.startswith("ATOM") or line[12:16].strip() != "CA":
            continue
        if chain_id is not None and line[21] != chain_id:
            continue
        resseq = int(line[22:26].strip())
        x, y, z = float(line[30:38]), float(line[38:46]), float(line[46:54])
        entries.append((resseq, (x, y, z)))

    entries.sort(key=lambda item: item[0])
    segments: list[list[tuple[float, float, float]]] = [[]]
    for _, coord in entries:
        if segments[-1]:
            prev = segments[-1][-1]
            if math.dist(prev, coord) > max_segment_angstrom:
                segments.append([])
        segments[-1].append(coord)
    return [seg for seg in segments if len(seg) >= 2]


def _pdb_receptor_heavy_coords(pdb_path: Path, *, ligand_chain: str = "B") -> list[tuple[float, float, float]]:
    """Heavy-atom coordinates for all receptor chains (everything except the ligand chain)."""
    coords: list[tuple[float, float, float]] = []
    for line in pdb_path.read_text().splitlines():
        if not line.startswith("ATOM"):
            continue
        if line[21] == ligand_chain:
            continue
        if _is_hydrogen(line[12:16]):
            continue
        coords.append((float(line[30:38]), float(line[38:46]), float(line[46:54])))
    return coords


def render_md_final_png(
    pdb_path: Path,
    out_path: Path,
    *,
    width: int = 1200,
    height: int = 900,
    dpi: int = 200,
) -> Path:
    """Static red/blue figure: receptor pocket as heavy-atom cloud, calx-β as CA trace."""
    receptor_pts = _pdb_receptor_heavy_coords(pdb_path)
    ligand_segments = _pdb_ca_traces(pdb_path, chain_id="B")
    if not receptor_pts and not ligand_segments:
        raise ValueError(f"No protein atoms found in {pdb_path.name}")

    fig = plt.figure(figsize=(width / dpi, height / dpi), dpi=dpi, facecolor="white")
    ax = fig.add_subplot(111, projection="3d")

    all_pts: list[tuple[float, float, float]] = list(receptor_pts)
    for seg in ligand_segments:
        all_pts.extend(seg)

    arr = np.array(all_pts)
    center = arr.mean(axis=0)
    keep_radius = max(15.0, np.linalg.norm(arr - center, axis=1).max() * 0.55)

    if receptor_pts:
        rec_arr = np.array(receptor_pts)
        keep = np.linalg.norm(rec_arr - center, axis=1) <= keep_radius
        rec_arr = rec_arr[keep]
        ax.scatter(
            rec_arr[:, 0],
            rec_arr[:, 1],
            rec_arr[:, 2],
            c="#e41a1c",
            s=10,
            alpha=0.55,
            depthshade=True,
            edgecolors="none",
        )

    for seg in ligand_segments:
        seg_arr = np.array(seg)
        if np.linalg.norm(seg_arr.mean(axis=0) - center) > keep_radius:
            continue
        xs, ys, zs = zip(*seg)
        ax.plot(xs, ys, zs, color="#377eb8", linewidth=2.2, alpha=0.95)

    arr = np.array([p for p in all_pts if np.linalg.norm(np.array(p) - center) <= keep_radius])
    center = arr.mean(axis=0)
    radius = max(1.0, (arr.max(axis=0) - arr.min(axis=0)).max() / 2)
    for axis, setter in enumerate([ax.set_xlim, ax.set_ylim, ax.set_zlim]):
        setter(center[axis] - radius, center[axis] + radius)

    ax.set_axis_off()
    ax.view_init(elev=22, azim=-65)
    fig.tight_layout(pad=0.2)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_path, bbox_inches="tight", pad_inches=0.05, facecolor="white")
    plt.close(fig)
    return out_path


def display_md_final_structure(pdb_path: Path, png_path: Path | None = None):
    """Inline PNG of MD final frame (nglview cartoon if available, else matplotlib fallback)."""
    png_path = png_path or pdb_path.with_suffix(".png")
    pocket_note = (
        "Receptor (red) = pocket fragments from the MD box (chains A,C,E,F,G,H,J,M); "
        "calx-β (blue) = chain B."
    )

    try:
        view = make_md_final_view(pdb_path)
        image_widget = view.render_image()
        for _ in range(100):
            if image_widget.value:
                break
            time.sleep(0.1)
        if image_widget.value:
            png_path.write_bytes(image_widget.value)
            display(Image(data=image_widget.value, format="png"))
            print(f"Saved (nglview cartoon): {png_path}")
            print(pocket_note)
            return
    except Exception as exc:
        print(f"nglview snapshot unavailable ({exc}); using matplotlib fallback.")

    render_md_final_png(pdb_path, png_path)
    display(Image(filename=str(png_path)))
    print(f"Saved: {png_path}")
    print(pocket_note)


## Structures

Each panel is one file from `data/3d_struct/`. Large receptors use **cartoon**; docked ligands use **ball+stick**.

In [ ]:
for path in STRUCT_FILES:
    if "IVM" not in str(path):
        meta = structure_summary(path)
        header = f"**{path.name}**"
        if meta["docked_energy_kcal_mol"] is not None:
            header += f" — docked energy {meta['docked_energy_kcal_mol']:+.2f} kcal/mol"
        display(Markdown(f"#### {header}"))
        if meta["warnings"]:
            print("note:", meta["warnings"])
        print(
            f"atoms={meta['n_atoms']}  residues={meta['n_residues']}  "
            f"models={meta['n_models']}"
        )
        display(make_view(path))


**This notebook is set for a 100 ns production run** (`TOTAL_STEPS = 50_000_000` at 2 fs). Production `mdrun` uses GPU offload (`-nb gpu -pme gpu -bonded gpu -update gpu`) when `USE_GPU_MDRUN = True` — run the kernel in the CUDA image (`./md-run.sh --gpu`). Equilibration (EM/NVT/NPT) runs on CPU `mdrun` under `data/md_calx_3qqu/` (gcsfuse). **Production** `grompp` + `mdrun` run under **`/tmp/md_calx_3qqu/`** (local disk); outputs are copied back to `data/md_calx_3qqu/` when the run finishes (and after each checkpoint write).

### Why `/tmp` for production
The workspace path is backed by **gcsfuse**. Long production runs write large, frequent files (`md.xtc`, `md.cpt`, `md.edr`, `md.log`). On gcsfuse these writes are slow and can block the Jupyter kernel (sometimes seen as `KeyboardInterrupt` during VM teardown). Local `/tmp` avoids that I/O path for the hours-long production step.

### Checkpoints every 1 ns (`nstcheckpoint = 500000`)
At `dt = 0.002` ps (2 fs), 1 ns = 1000 ps → **500000 steps** between checkpoint writes. GROMACS keeps a single `md.cpt` file and overwrites it each time; the default interval (~15 min wall time, ~40k steps) caused too many gcsfuse writes when production ran on the workspace. With production on `/tmp`, checkpoints still use a 1 ns interval so a VM loss loses at most ~1 ns of simulation time.

**Re-grompp for a new checkpoint interval:** `nstcheckpoint` is stored in `md.tpr` at `grompp` time. Editing `md.mdp` alone does not change an existing `md.tpr`. To use the 1 ns interval, re-run production `grompp` (pipeline step 7) before the next `mdrun`. You can still resume from an existing `md.cpt` with `-cpi md.cpt`; only the *future* checkpoint schedule comes from the new `.tpr`.

### Must re-run helper cells after any notebook sync
Jupyter keeps **old function bodies in the kernel** until you re-execute them. After pulling/syncing this file into `/workspace`, **re-run the MD setup cells** that define `sync_md_to_tmp` / `run_gmx` / `production_mdrun_args` / `run_md_pipeline` (the cells under this section) **before** the `RUN_MD` pipeline cell. Confirm the helper printout mentions `/tmp/md_calx_3qqu`.

Verbose GROMACS stdout is written under **`/tmp`** during long runs, with progress milestones only in the notebook; the log is copied into `data/runs/` when the pipeline finishes.

**Session idle / teardown:** If the launch UI shows "Idle timeout very soon" / remaining `0:00`, the VM may tear down soon and the kernel will raise `KeyboardInterrupt` even while `mdrun` is progressing. Extend idle time from the session control bar and confirm remaining time is non-zero before starting a multi-hour production run.

### Re-entrant `RUN_MD` (automatic resume)
When **`data/md_calx_3qqu/md.cpt`** exists in the workspace (GCS via gcsfuse), re-run the helper cells and the **`RUN_MD` cell** — it **does not** repeat solvation, EM, NVT, or NPT. It:

1. Copies the latest checkpoint and matching inputs from workspace → **`/tmp/md_calx_3qqu/`**
2. Runs production `mdrun` with **`-cpi md.cpt`** on local disk (GPU flags when enabled)
3. Copies **`md.cpt` back to the workspace after each checkpoint write** (~every 1 ns), then full outputs at the end

Set **`FORCE_FROM_SCRATCH = True`** in the config cell to ignore an existing checkpoint and run the full pipeline from solvation.

**Do not re-run production `grompp` on resume** — that would restart from NPT coordinates instead of the checkpoint. Resume uses the existing **`md.tpr`** paired with **`md.cpt`**. To change `nstcheckpoint` after a run has started, finish or discard the run and start fresh with `FORCE_FROM_SCRATCH = True`.

In [ ]:
@dataclass
class MdPaths:
    root: Path

    @property
    def receptor_clean(self) -> Path:
        return self.root / "receptor_3QQU_chainA.pdb"

    @property
    def ligand_clean(self) -> Path:
        return self.root / "ligand_calxbeta_chainB.pdb"

    @property
    def complex_pdb(self) -> Path:
        return self.root / "complex_calx_3qqu.pdb"

    @property
    def pocket_pdb(self) -> Path:
        return self.root / "pocket_calx_3qqu.pdb"

    @property
    def processed_gro(self) -> Path:
        return self.root / "pocket_processed.gro"

    @property
    def topol_top(self) -> Path:
        return self.root / "topol.top"


MD = MdPaths(MD_ROOT)


def _read_md_structure_text(path: Path) -> str:
    """ATOM/HETATM records for GROMACS/mdtraj (full coords; no nglview 80-col trim)."""
    lines = [
        line
        for line in path.read_text(errors="replace").splitlines()
        if line.startswith(("ATOM", "HETATM"))
    ]
    return "\n".join(lines) + "\n"


def _normalize_atom_name(raw: str) -> str:
    """Map AutoDock/MGL atom names to PDB names expected by pdb2gmx."""
    name = raw.strip()
    return {
        "AG": "CG",
        "AD1": "CD1",
        "AD2": "CD2",
        "AE1": "CE1",
        "AE2": "CE2",
        "AE3": "CE3",
        "AZ": "CZ",
        "AZ2": "CZ2",
        "AZ3": "CZ3",
        "AH2": "CH2",
        "HN": "H",
    }.get(name, name)


def _pdb_atom_field(raw: str) -> str:
    name = _normalize_atom_name(raw)
    if len(name) >= 4 or (len(name) == 3 and name[0].isdigit()):
        return name[:4].ljust(4)
    return name.rjust(4)


def _format_pdb_atom_line(line: str, serial: int, chain_id: str) -> str | None:
    """Emit a strict PDB v3 ATOM line (mdtraj-safe fixed-width columns)."""
    if len(line) < 54:
        return None
    record = line[:6]
    atom_field = _pdb_atom_field(line[12:16])
    if not atom_field.strip():
        return None
    alt_loc = line[16] if len(line) > 16 else " "
    resname = line[17:20].strip() or "UNK"
    if resname in {"HOH", "WAT", "SOL"}:
        return None

    resseq_str = line[22:26].strip() or line[21:25].strip()
    resseq = int(resseq_str)
    icode = line[26] if len(line) > 26 else " "

    x = float(line[30:38])
    y = float(line[38:46])
    z = float(line[46:54])
    occ = float(line[54:60]) if len(line) >= 60 and line[54:60].strip() else 1.0
    bf = float(line[60:66]) if len(line) >= 66 and line[60:66].strip() else 0.0

    return (
        f"{record[:6]:<6}{serial:5d} {atom_field}{alt_loc}{resname:>3} {chain_id}"
        f"{resseq:4d}{icode}   {x:8.3f}{y:8.3f}{z:8.3f}{occ:6.2f}{bf:6.2f}\n"
    )


def _atom_lines(text: str, chain_id: str) -> list[str]:
    out: list[str] = []
    serial = 1
    for line in text.splitlines():
        if not line.startswith(("ATOM", "HETATM")):
            continue
        formatted = _format_pdb_atom_line(line, serial, chain_id)
        if formatted is None:
            continue
        out.append(formatted)
        serial += 1
    return out


def _renumber_atom_serials(lines: list[str], start: int = 1) -> list[str]:
    out: list[str] = []
    serial = start
    for line in lines:
        out.append(f"{line[:6]}{serial:5d}{line[11:]}")
        serial += 1
    return out


def build_docked_complex() -> Path:
    """Merge receptor (chain A) + docked calx-β (chain B) into one PDB."""
    rec_text = _read_md_structure_text(RECEPTOR_PATH)
    lig_text = _read_md_structure_text(LIGAND_PATH)

    rec_lines = _atom_lines(rec_text, "A")
    lig_lines = _atom_lines(lig_text, "B")
    complex_lines = _renumber_atom_serials(rec_lines + lig_lines)

    MD.receptor_clean.write_text("".join(rec_lines))
    MD.ligand_clean.write_text("".join(lig_lines))

    with MD.complex_pdb.open("w") as handle:
        handle.write("".join(complex_lines))
        handle.write("TER\n")
        handle.write("END\n")

    print(f"complex: {MD.complex_pdb}  atoms={len(complex_lines)}")
    return MD.complex_pdb


def _pocket_chain_segments(traj: md.Trajectory, keep: list[int]) -> list[tuple[str, set[int]]]:
    """Split kept atoms into sequence-contiguous segments with unique chain IDs."""
    segments: list[tuple[str, set[int]]] = []
    current_chain_idx: int | None = None
    current_res_idx: int | None = None
    current_resseq: int | None = None
    current_atoms: set[int] = set()
    next_rec_chain_ord = ord("A")

    def flush() -> None:
        nonlocal current_atoms, current_chain_idx, current_res_idx, current_resseq, next_rec_chain_ord
        if not current_atoms:
            return
        if current_chain_idx == 1:
            chain_id = "B"
        else:
            while chr(next_rec_chain_ord) in {"B"}:
                next_rec_chain_ord += 1
            chain_id = chr(next_rec_chain_ord)
            next_rec_chain_ord += 1
        segments.append((chain_id, current_atoms))
        current_atoms = set()
        current_chain_idx = None
        current_res_idx = None
        current_resseq = None

    for atom_idx in keep:
        res = traj.top.atom(atom_idx).residue
        chain_idx = res.chain.index
        same_residue = current_res_idx is not None and res.index == current_res_idx
        next_in_sequence = current_resseq is not None and res.resSeq == current_resseq + 1
        contiguous = current_chain_idx == chain_idx and (same_residue or next_in_sequence)
        if current_atoms and not contiguous:
            flush()
        current_chain_idx = chain_idx
        current_res_idx = res.index
        current_resseq = res.resSeq
        current_atoms.add(atom_idx)

    flush()
    return segments


def extract_binding_pocket(complex_pdb: Path, *, cutoff_nm: float = POCKET_CUTOFF_NM) -> Path:
    """Keep calx-β and IGF1R residues within cutoff of the peptide."""
    traj = md.load_pdb(str(complex_pdb))
    lig = traj.top.select("chainid B")
    if lig.size == 0 and traj.top.n_chains >= 2:
        lig = np.array([a.index for a in traj.top.atoms if a.residue.chain.index == 1])
    if lig.size == 0:
        n_rec = sum(
            1
            for line in MD.receptor_clean.read_text().splitlines()
            if line.startswith(("ATOM", "HETATM"))
        )
        if n_rec < traj.n_atoms:
            lig = np.arange(n_rec, traj.n_atoms)
    if lig.size == 0:
        raise ValueError("No calx-β atoms found in complex")

    near = md.compute_neighbors(traj, cutoff_nm, query_indices=lig)[0]
    seed = set(lig.tolist()) | set(near.tolist())
    # pdb2gmx needs complete residues — expand atom cutoff hits to whole residues.
    residue_indices = {traj.top.atom(i).residue.index for i in seed}
    keep = sorted(a.index for a in traj.top.atoms if a.residue.index in residue_indices)

    atom_lines = [
        line
        for line in complex_pdb.read_text().splitlines()
        if line.startswith(("ATOM", "HETATM"))
    ]
    if len(atom_lines) != traj.n_atoms:
        raise ValueError("Complex PDB atom count does not match mdtraj topology")

    segments = _pocket_chain_segments(traj, keep)
    # pdb2gmx cannot parameterize a one-residue chain (no standalone ARG/LYS/… termini).
    kept_segments: list[tuple[str, set[int]]] = []
    pruned: list[str] = []
    for chain_id, idxs in segments:
        res_in_seg = {traj.top.atom(i).residue.index for i in idxs}
        if len(res_in_seg) < 2:
            res = traj.top.atom(next(iter(idxs))).residue
            pruned.append(f"{chain_id}:{res.name}{res.resSeq}")
            continue
        kept_segments.append((chain_id, idxs))
    if pruned:
        print(f"skipped single-residue fragments: {', '.join(pruned)}")
    segments = kept_segments
    atom_to_chain = {idx: chain_id for chain_id, idxs in segments for idx in idxs}

    serial = 1
    with MD.pocket_pdb.open("w") as handle:
        for chain_id, idxs in segments:
            for i in sorted(idxs):
                formatted = _format_pdb_atom_line(atom_lines[i], serial, atom_to_chain[i])
                if formatted is None:
                    continue
                handle.write(formatted)
                serial += 1
            handle.write("TER\n")
        handle.write("END\n")

    n_lig = len(set(lig.tolist()) & set(keep))
    n_rec = len(keep) - n_lig
    n_res = len(residue_indices)
    print(
        f"pocket: {MD.pocket_pdb}  atoms={serial - 1} residues={n_res} chains={len(segments)} "
        f"(receptor={n_rec}, calx-β={n_lig}, cutoff={cutoff_nm:.1f} nm)"
    )
    return MD.pocket_pdb


def save_md_final_protein_pdb(traj: md.Trajectory, pocket_pdb: Path, out_path: Path) -> Path:
    """Write final MD coords into the pocket topology (keeps chain IDs and PDB columns)."""
    frame = traj[-1].atom_slice(traj.top.select("protein"))
    coord_map: dict[tuple[str, int, str], np.ndarray] = {}
    for atom in frame.top.atoms:
        key = (atom.residue.name, atom.residue.resSeq, atom.name.strip())
        coord_map[key] = frame.xyz[0, atom.index]

    lines: list[str] = []
    serial = 1
    for line in pocket_pdb.read_text().splitlines():
        if not line.startswith(("ATOM", "HETATM")):
            continue
        resname = line[17:20].strip()
        resseq_str = line[22:26].strip() or line[21:25].strip()
        atom_name = line[12:16].strip()
        key = (resname, int(resseq_str), atom_name)
        if key not in coord_map:
            continue
        # mdtraj stores nm; PDB expects Å.
        x, y, z = (coord_map[key] * 10.0).tolist()
        chain_id = line[21]
        base = line.ljust(66)
        patched = f"{base[:30]}{x:8.3f}{y:8.3f}{z:8.3f}{1.0:6.2f}{0.0:6.2f}\n"
        formatted = _format_pdb_atom_line(patched, serial, chain_id)
        if formatted is None:
            continue
        lines.append(formatted)
        serial += 1

    out_path.write_text("".join(lines) + "TER\nEND\n")
    return out_path


def md_results_manifest_path() -> Path:
    return MD.root / "md_results_manifest.json"


def save_md_results_manifest(*, process_ok: bool) -> Path | None:
    """Persist manifest pointing at on-disk MD analysis outputs (only if process_ok)."""
    if not process_ok:
        print("Skipping manifest save (processing did not succeed this run).")
        return None

    final_pdb = MD.root / "md_final_protein.pdb"
    if not final_pdb.exists():
        print("Skipping manifest save (md_final_protein.pdb missing).")
        return None

    ener_xvg = MD.root / "md_ener.xvg"
    manifest = {
        "ok": True,
        "saved_at": datetime.now(timezone.utc).isoformat(),
        "final_pdb": str(final_pdb.resolve()),
        "final_png": str((MD.root / "md_final_protein.png").resolve()),
        "energy_xvg": str(ener_xvg.resolve()) if ener_xvg.exists() else None,
        "trajectory": str((MD.root / "md.xtc").resolve()),
        "pbc_trajectory": str((MD.root / "md_pbc.xtc").resolve())
        if (MD.root / "md_pbc.xtc").exists()
        else None,
        "topology": str((MD.root / "npt.gro").resolve()),
    }
    path = md_results_manifest_path()
    path.write_text(json.dumps(manifest, indent=2) + "\n")
    print(f"Saved MD results manifest: {path}")
    return path


def load_md_results_manifest() -> dict | None:
    path = md_results_manifest_path()
    if not path.exists():
        return None

    return json.loads(path.read_text())


def _manifest_artifact_path(key: str, default: Path) -> Path:
    """Resolve manifest artifact paths; fall back to MD.root when stale absolute paths."""
    manifest = load_md_results_manifest()
    if not manifest or not manifest.get(key):
        return default
    stored = Path(manifest[key])
    if not stored.is_absolute():
        relative = default.parent / stored
        if relative.exists():
            return relative
    if stored.exists():
        return stored
    by_name = default.parent / stored.name
    if by_name.exists():
        return by_name
    return default


def plot_md_energy_xvg(xvg_path: Path) -> bool:
    """Plot potential energy from a GROMACS .xvg file."""
    rows: list[tuple[float, float]] = []
    for line in xvg_path.read_text().splitlines():
        if line.startswith(("#", "@")):
            continue
        parts = line.split()
        if len(parts) == 2:
            rows.append((float(parts[0]), float(parts[1])))
    if not rows:
        return False

    df = pd.DataFrame(rows, columns=["time_ps", "potential_kJ_mol"])
    df.plot(x="time_ps", y="potential_kJ_mol", figsize=(8, 3), legend=False)
    plt.ylabel("Potential energy (kJ/mol)")
    plt.xlabel("Time (ps)")
    plt.title("Calx-β / IGF1R pocket — production MD")
    plt.tight_layout()
    plt.show()
    plt.close("all")
    return True


def md_binding_metrics_path() -> Path:
    return MD.root / "md_binding_metrics.csv"


def ensure_pbc_corrected_trajectory(
    *,
    xtc: Path | None = None,
    tpr: Path | None = None,
    out_xtc: Path | None = None,
    log_path: Path | None = None,
    force: bool = False,
) -> Path:
    """Make molecules whole and center on protein (gmx trjconv -pbc mol -center)."""
    xtc = xtc or MD.root / "md.xtc"
    tpr = tpr or MD.root / "md.tpr"
    out_xtc = out_xtc or MD.root / "md_pbc.xtc"
    if not xtc.exists():
        raise FileNotFoundError(f"Missing trajectory: {xtc}")
    if not force and out_xtc.exists() and out_xtc.stat().st_mtime >= xtc.stat().st_mtime:
        return out_xtc
    if not tpr.exists():
        raise FileNotFoundError(f"Missing {tpr.name} — run production grompp/mdrun first.")
    print(f"PBC correction: {xtc.name} → {out_xtc.name} (-pbc mol -center)", flush=True)
    run_gmx(
        [
            "trjconv",
            "-s", tpr.name,
            "-f", xtc.name,
            "-o", out_xtc.name,
            "-pbc", "mol",
            "-center",
            "-ur", "compact",
        ],
        stdin="Protein\nSystem\n",
        log_path=log_path,
    )
    return out_xtc


def load_md_trajectory(
    *,
    apply_pbc: bool = True,
    force_pbc: bool = False,
    log_path: Path | None = None,
) -> md.Trajectory:
    """Load production trajectory (+ npt.gro); optionally PBC-correct via md_pbc.xtc."""
    manifest = load_md_results_manifest()
    xtc = _manifest_artifact_path("trajectory", MD.root / "md.xtc")
    gro = _manifest_artifact_path("topology", MD.root / "npt.gro")
    pbc_xtc = _manifest_artifact_path("pbc_trajectory", MD.root / "md_pbc.xtc")
    if not xtc.exists() or not gro.exists():
        raise FileNotFoundError("Missing md.xtc or npt.gro — run GROMACS first.")
    if apply_pbc:
        xtc = ensure_pbc_corrected_trajectory(
            xtc=xtc, out_xtc=pbc_xtc, log_path=log_path, force=force_pbc,
        )
    return md.load(str(xtc), top=str(gro))


def _pocket_residue_keys(pocket_pdb: Path) -> tuple[set[tuple[str, int]], set[tuple[str, int]]]:
    """Return (calx-β, receptor) residue keys as (resname, resSeq) — chain IDs are lost in .gro."""
    lig_res: set[tuple[str, int]] = set()
    rec_res: set[tuple[str, int]] = set()
    for line in pocket_pdb.read_text().splitlines():
        if not line.startswith("ATOM"):
            continue
        key = (line[17:20].strip(), int(line[22:26].strip()))
        if line[21] == "B":
            lig_res.add(key)
        else:
            rec_res.add(key)
    return lig_res, rec_res


def _indices_for_residue_keys(traj: md.Trajectory, keys: set[tuple[str, int]], *, heavy_only: bool) -> list[int]:
    idx: list[int] = []
    for atom in traj.top.atoms:
        res = atom.residue
        if (res.name, res.resSeq) not in keys:
            continue
        if heavy_only and atom.element.symbol == "H":
            continue
        idx.append(atom.index)
    return idx


def compute_binding_metrics(
    traj: md.Trajectory,
    pocket_pdb: Path,
    *,
    contact_cutoff_nm: float = 0.4,
) -> pd.DataFrame:
    """Calx-β RMSD (nm, vs frame 0) and receptor contact counts per frame."""
    lig_res, rec_res = _pocket_residue_keys(pocket_pdb)
    lig_idx = _indices_for_residue_keys(traj, lig_res, heavy_only=True)
    rec_idx = _indices_for_residue_keys(traj, rec_res, heavy_only=True)
    if not lig_idx or not rec_idx:
        raise ValueError("Could not map pocket calx-β / receptor residues onto trajectory.")

    lig_traj = traj.atom_slice(lig_idx)
    lig_traj.superpose(lig_traj, 0)
    rmsd_nm = md.rmsd(lig_traj, lig_traj, 0)

    rec_idx_set = set(rec_idx)
    rec_by_res = {a.index: a.residue.index for a in traj.top.atoms if a.index in rec_idx_set}
    n_contacts: list[int] = []
    n_contact_res: list[int] = []
    for neighbors in md.compute_neighbors(traj, contact_cutoff_nm, query_indices=lig_idx):
        rec_neighbors = set(neighbors) & rec_idx_set
        n_contacts.append(len(rec_neighbors))
        n_contact_res.append(len({rec_by_res[i] for i in rec_neighbors}))

    time_ps = traj.time if len(traj.time) == traj.n_frames else np.arange(traj.n_frames, dtype=float)

    return pd.DataFrame(
        {
            "time_ps": time_ps,
            "ligand_rmsd_nm": rmsd_nm,
            "receptor_contacts": n_contacts,
            "receptor_contact_residues": n_contact_res,
        }
    )


def save_binding_metrics(metrics: pd.DataFrame) -> Path:
    out = md_binding_metrics_path()
    metrics.to_csv(out, index=False)
    print(f"Saved binding metrics: {out}")
    return out


def load_binding_metrics() -> pd.DataFrame | None:
    path = md_binding_metrics_path()
    if not path.exists():
        return None
    return pd.read_csv(path)


def plot_binding_metrics(metrics: pd.DataFrame) -> None:
    fig, axes = plt.subplots(2, 1, figsize=(8, 5), sharex=True)
    metrics.plot(x="time_ps", y="ligand_rmsd_nm", ax=axes[0], legend=False, color="#377eb8")
    axes[0].set_ylabel("Calx-β RMSD (nm)")
    axes[0].set_title("Ligand stability in pocket (aligned to frame 0)")
    metrics.plot(
        x="time_ps",
        y=["receptor_contacts", "receptor_contact_residues"],
        ax=axes[1],
        color=["#e41a1c", "#984ea3"],
    )
    axes[1].set_ylabel("Contacts (≤ 0.4 nm)")
    axes[1].set_xlabel("Time (ps)")
    axes[1].set_title("Receptor–calx-β contacts")
    axes[1].legend(["Receptor heavy atoms", "Receptor residues"], loc="best")
    fig.tight_layout()
    plt.show()
    plt.close("all")


def assess_binding_stability(
    metrics: pd.DataFrame,
    *,
    docked_kcal_mol: float = -18.39,
    stable_rmsd_nm: float = 0.25,
    moderate_rmsd_nm: float = 0.50,
) -> str:
    """Heuristic summary from RMSD + contact persistence (not binding free energy)."""
    rmsd = metrics["ligand_rmsd_nm"].to_numpy()
    contacts = metrics["receptor_contact_residues"].to_numpy()
    atom_contacts = metrics["receptor_contacts"].to_numpy()
    t = metrics["time_ps"].to_numpy()
    traj_ps = float(t[-1] - t[0]) if len(t) > 1 else float(t[-1]) if len(t) else 0.0
    MIN_AFFINITY_MD_PS = 10_000.0  # 10 ns — below this, pose-only conclusions
    STRONG_MD_PS = 100_000.0  # 100 ns — aligns with footnote guidance

    median_r = float(np.median(rmsd))
    max_r = float(np.max(rmsd))
    late = rmsd[int(0.75 * len(rmsd)) :]
    early = rmsd[: max(1, int(0.25 * len(rmsd)))]
    drift = float(np.median(late) - np.median(early))

    med_c = float(np.median(contacts))
    min_c = int(np.min(contacts))
    late_c = contacts[int(0.75 * len(contacts)) :]
    med_atoms = float(np.median(atom_contacts))

    if max_r < stable_rmsd_nm and drift < 0.05:
        rmsd_verdict = "**stable in the docked pose**"
    elif max_r < moderate_rmsd_nm and drift < 0.15:
        rmsd_verdict = "**moderately mobile but largely retained**"
    elif med_c >= 8:
        rmsd_verdict = "**substantial rearrangement within the pocket** (contacts retained)"
    else:
        rmsd_verdict = "**significant displacement** with weakening contacts (possible partial unbinding)"

    if med_c >= 12 and min_c >= 8 and np.median(late_c) >= 0.75 * med_c:
        contact_verdict = "Receptor contacts are **persistent** across the run."
    elif med_c >= 6:
        contact_verdict = "Receptor contacts are **partially maintained** (some fluctuation)."
    else:
        contact_verdict = "Receptor contacts are **sparse or declining** — binding may not be sustained."

    lines = [
        "### Binding stability assessment (heuristic)",
        "",
        f"- **Trajectory length:** {traj_ps:.0f} ps ({len(metrics)} frames)",
        f"- **Static dock score (Vina):** {docked_kcal_mol:+.2f} kcal/mol (pose quality only, not MD ΔG)",
        f"- **Calx-β RMSD vs t=0:** median {median_r:.2f} nm, max {max_r:.2f} nm, late−early drift {drift:+.2f} nm → {rmsd_verdict}",
        f"- **Receptor contacts (≤ 0.4 nm):** median {med_atoms:.0f} heavy-atom pairs / {med_c:.0f} residues, min {min_c} residues → {contact_verdict}",
        "",
        "**Overall:** ",
    ]

    if max_r < moderate_rmsd_nm and med_c >= 8 and drift < 0.15:
        overall = (
            "The complex appears **relatively stable** over this production run: "
            "calx-β stays near the docked pose with sustained pocket contacts. "
            "This supports **retained binding**"
        )
        if traj_ps < MIN_AFFINITY_MD_PS:
            overall += (
                f", but {traj_ps:.0f} ps is too brief for definitive affinity conclusions."
            )
        elif traj_ps < STRONG_MD_PS:
            overall += (
                f" over {traj_ps / 1000:.1f} ns, though quantitative affinity still "
                "requires MM-PBSA/GBSA or longer sampling."
            )
        else:
            overall += (
                f" over {traj_ps / 1000:.1f} ns; MM-PBSA/GBSA is still required for "
                "quantitative ΔG_bind."
            )
    elif med_c >= 10 and min_c >= 6:
        overall = (
            "Calx-β shows **appreciable internal motion or pocket rearrangement** (elevated RMSD), "
            "but **receptor contact counts stay high** — consistent with **binding retained in the site** "
            "rather than dissociation. "
        )
        if traj_ps < MIN_AFFINITY_MD_PS:
            overall += (
                "Longer MD would clarify whether the pose equilibrates or drifts further."
            )
        elif traj_ps < STRONG_MD_PS:
            overall += (
                f"Over {traj_ps / 1000:.1f} ns, extended sampling or MM-PBSA/GBSA would "
                "clarify whether the pose equilibrates or drifts further."
            )
        else:
            overall += (
                "Inspect the trajectory and consider MM-PBSA/GBSA for quantitative affinity."
            )
    elif max_r < moderate_rmsd_nm:
        overall = (
            "Calx-β **remains in the general pocket region**, but contact patterns suggest "
            "**partial mobility or rearrangement**. Treat as **plausible but not strongly confirmed** binding."
        )
    else:
        overall = (
            "Large ligand RMSD together with weak or declining contacts suggests **unstable or lost binding** "
            "relative to the docked starting pose — inspect the trajectory visually before drawing conclusions."
        )

    lines.append(overall)
    lines.append("")
    if traj_ps < MIN_AFFINITY_MD_PS:
        note = (
            "_Note: Total potential energy and these metrics do not equal binding free energy. "
            "Longer MD (≥10–100 ns) and interaction energy (MM-PBSA/GBSA) would be needed for "
            "stronger statements._"
        )
    elif traj_ps < STRONG_MD_PS:
        note = (
            "_Note: These metrics do not equal binding free energy. MM-PBSA/GBSA (or longer "
            "sampling toward ~100 ns) would be needed for quantitative affinity._"
        )
    else:
        note = (
            "_Note: RMSD and contact metrics do not equal binding free energy. MM-PBSA/GBSA is "
            "still required for quantitative ΔG_bind._"
        )
    lines.append(note)

    text = "\n".join(lines)
    display(Markdown(text))
    return text


def display_binding_analysis() -> bool:
    """Plot saved binding metrics and print assessment (no trajectory load)."""
    metrics = load_binding_metrics()
    if metrics is None or metrics.empty:
        print("No md_binding_metrics.csv — run the binding analysis cell first.")
        return False
    plot_binding_metrics(metrics)
    assess_binding_stability(metrics)
    return True


def _saved_md_result_paths() -> tuple[Path, Path, Path]:
    """Resolve final PDB, PNG, and energy XVG from manifest or defaults."""
    manifest = load_md_results_manifest()
    final_pdb = _manifest_artifact_path("final_pdb", MD.root / "md_final_protein.pdb")
    final_png = _manifest_artifact_path("final_png", MD.root / "md_final_protein.png")
    ener_xvg = _manifest_artifact_path("energy_xvg", MD.root / "md_ener.xvg")
    return final_pdb, final_png, ener_xvg


def display_saved_md_energy() -> bool:
    """Plot total potential energy from saved md_ener.xvg (no trajectory load)."""
    manifest = load_md_results_manifest()
    if manifest:
        print(f"Manifest: {md_results_manifest_path()} (saved {manifest.get('saved_at', '?')})")

    _, _, ener_xvg = _saved_md_result_paths()
    if not ener_xvg.exists():
        print("No md_ener.xvg — skipping energy plot.")
        return False

    plot_md_energy_xvg(ener_xvg)
    return True


build_docked_complex()
extract_binding_pocket(MD.complex_pdb)
display(make_view(MD.pocket_pdb))


In [ ]:
def _write_mdp(path: Path, text: str) -> None:
    path.write_text(text.strip() + "\n")


def write_mdp_files() -> dict[str, Path]:
    mdps = {
        "ions": MD.root / "ions.mdp",
        "em": MD.root / "em.mdp",
        "nvt": MD.root / "nvt.mdp",
        "npt": MD.root / "npt.mdp",
        "md": MD.root / "md.mdp",
    }
    _write_mdp(
        mdps["ions"],
        """
integrator  = steep
emtol       = 1000.0
emstep      = 0.01
nsteps      = 1
nstlist     = 1
cutoff-scheme = Verlet
coulombtype = PME
rcoulomb    = 1.0
rvdw        = 1.0
""",
    )
    em_log_interval = max(1, EM_STEPS // 100)
    _write_mdp(
        mdps["em"],
        f"""
integrator  = steep
emtol       = 1000.0
emstep      = 0.01
nsteps      = {EM_STEPS}
nstlist     = {em_log_interval}
cutoff-scheme = Verlet
coulombtype = PME
rcoulomb    = 1.0
rvdw        = 1.0
""",
    )
    nvt_log_interval = max(1, NVT_STEPS // 100)
    _write_mdp(
        mdps["nvt"],
        f"""
define      = -DPOSRES
integrator  = md
nsteps      = {NVT_STEPS}
dt          = {MD_DT_PS}
nstxout     = 0
nstvout     = 0
nstenergy   = {nvt_log_interval}
nstlog      = {nvt_log_interval}
continuation = no
constraint_algorithm = lincs
constraints = h-bonds
cutoff-scheme = Verlet
coulombtype = PME
rcoulomb    = 1.0
rvdw        = 1.0
tcoupl      = V-rescale
tc-grps     = Protein Non-Protein
tau_t       = 0.1 0.1
ref_t       = 300 300
""",
    )
    npt_log_interval = max(1, NPT_STEPS // 100)
    _write_mdp(
        mdps["npt"],
        f"""
define      = -DPOSRES
integrator  = md
nsteps      = {NPT_STEPS}
dt          = {MD_DT_PS}
nstxout     = 0
nstvout     = 0
nstenergy   = {npt_log_interval}
nstlog      = {npt_log_interval}
continuation = yes
constraint_algorithm = lincs
constraints = h-bonds
cutoff-scheme = Verlet
coulombtype = PME
rcoulomb    = 1.0
rvdw        = 1.0
tcoupl      = V-rescale
tc-grps     = Protein Non-Protein
tau_t       = 0.1 0.1
ref_t       = 300 300
pcoupl      = Parrinello-Rahman
pcoupltype  = isotropic
tau_p       = 2.0
ref_p       = 1.0
compressibility = 4.5e-5
""",
    )
    progress_interval = max(1, TOTAL_STEPS // 100)
    traj_save_steps = max(1, int(round(TRAJ_SAVE_PS / MD_DT_PS)))
    _write_mdp(
        mdps["md"],
        f"""
integrator  = md
nsteps      = {TOTAL_STEPS}
dt          = {MD_DT_PS}
nstcheckpoint = {CHECKPOINT_SAVE_STEPS}
nstxout-compressed = {traj_save_steps}
compressed-x-grps  = System
nstenergy   = {traj_save_steps}
nstlog      = {progress_interval}
continuation = yes
constraint_algorithm = lincs
constraints = h-bonds
cutoff-scheme = Verlet
coulombtype = PME
rcoulomb    = 1.0
rvdw        = 1.0
tcoupl      = V-rescale
tc-grps     = Protein Non-Protein
tau_t       = 0.1 0.1
ref_t       = 300 300
pcoupl      = Parrinello-Rahman
pcoupltype  = isotropic
tau_p       = 2.0
ref_p       = 1.0
compressibility = 4.5e-5
""",
    )
    return mdps


_MD_TMP_TOPOLOGY_NAMES = ("topol.top", "posre.itp")
_MD_TMP_EQUIL_NAMES = ("npt.gro", "npt.cpt", "em.gro", "nvt.gro", "nvt.cpt")
_MD_TMP_PRODUCTION_NAMES = ("md.tpr", "md.cpt", "md.xtc", "md.edr", "md.log", "md.gro")
_MD_TMP_SYNC_BACK = ("md.cpt", "md.xtc", "md.edr", "md.log", "md.tpr", "md.gro")


def sync_md_to_tmp() -> Path:
    """Copy GROMACS inputs from workspace MD_ROOT to local MD_TMP_ROOT for production.

    Equilibration outputs and topology must exist under data/md_calx_3qqu/ first.
    Existing md.cpt (resume) is copied when present.
    """
    MD_TMP_ROOT.mkdir(parents=True, exist_ok=True)
    copied: list[str] = []

    def _copy(name: str) -> None:
        src = MD.root / name
        if src.is_file():
            shutil.copy2(src, MD_TMP_ROOT / name)
            copied.append(name)

    for name in _MD_TMP_TOPOLOGY_NAMES:
        _copy(name)
    for itp in sorted(MD.root.glob("*.itp")):
        if itp.name not in _MD_TMP_TOPOLOGY_NAMES:
            shutil.copy2(itp, MD_TMP_ROOT / itp.name)
            copied.append(itp.name)
    for mdp in MDPS.values():
        if mdp.is_file():
            shutil.copy2(mdp, MD_TMP_ROOT / mdp.name)
            copied.append(mdp.name)
    for name in _MD_TMP_EQUIL_NAMES + _MD_TMP_PRODUCTION_NAMES:
        _copy(name)

    print(
        f"Synced {len(copied)} file(s) → {MD_TMP_ROOT} "
        f"(production mdrun uses local disk, not gcsfuse)",
        flush=True,
    )
    return MD_TMP_ROOT


def sync_md_from_tmp(*, names: tuple[str, ...] = _MD_TMP_SYNC_BACK) -> list[Path]:
    """Copy production outputs from MD_TMP_ROOT back to workspace MD_ROOT."""
    MD.root.mkdir(parents=True, exist_ok=True)
    copied: list[Path] = []
    for name in names:
        src = MD_TMP_ROOT / name
        if src.is_file():
            dst = MD.root / name
            shutil.copy2(src, dst)
            copied.append(dst)
    if copied:
        print(
            "Copied from /tmp → workspace: " + ", ".join(p.name for p in copied),
            flush=True,
        )
    return copied


_STEP_RE = re.compile(r"\bstep\s+(\d+)", re.I)
_MD_RUN_LOG_ARCHIVE: Path | None = None


def open_md_run_log() -> Path:
    """Create a live run log under /tmp (not gcsfuse). Archive path is under data/runs/.

    Long mdrun must not append every stdout line to a /workspace (gcsfuse) file —
    that can stall the kernel and surface as KeyboardInterrupt during teardown.
    Call finalize_md_run_log() after the pipeline to copy into RUNS_DIR.
    """
    global _MD_RUN_LOG_ARCHIVE
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    live = Path("/tmp") / f"md_calx_3qqu_{ts}.log"
    archive = RUNS_DIR / f"md_calx_3qqu_{ts}.log"
    header = (
        f"=== GROMACS pipeline {ts} ===\n"
        f"EM_STEPS={EM_STEPS}  NVT_STEPS={NVT_STEPS}  NPT_STEPS={NPT_STEPS}\n"
        f"TOTAL_STEPS={TOTAL_STEPS}  dt={MD_DT_PS} ps  "
        f"production={TOTAL_STEPS * MD_DT_PS:.1f} ps\n"
        f"live_log={live}  archive={archive}\n\n"
    )
    live.write_text(header, encoding="utf-8")
    _MD_RUN_LOG_ARCHIVE = archive
    print(
        f"Live run log (local /tmp): {live}\n"
        f"Will copy to workspace at end: {archive}",
        flush=True,
    )
    return live


def finalize_md_run_log(live: Path | None) -> Path | None:
    """Best-effort copy of /tmp live log into data/runs/ (gcsfuse-safe, once)."""
    if live is None:
        return None
    archive = globals().get("_MD_RUN_LOG_ARCHIVE")
    if not isinstance(archive, Path):
        archive = RUNS_DIR / live.name
    try:
        archive.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(live, archive)
        print(f"Copied run log → {archive}", flush=True)
        return archive
    except OSError as exc:
        print(f"Warning: could not copy run log to workspace ({exc}); left at {live}", flush=True)
        return live


def run_gmx(
    args: list[str],
    *,
    cwd: Path = MD.root,
    stdin: str | None = None,
    stream: bool | None = None,
    log_path: Path | None = None,
    progress_interval: int | None = None,
    progress_total_steps: int | None = None,
) -> str:
    """Run gmx. For mdrun: full stdout → local log_path (/tmp); notebook gets progress only."""
    cmd = ["gmx", *args]
    print("$", " ".join(cmd), flush=True)
    if stream is None:
        stream = bool(args and args[0] == "mdrun")

    # Keep one file handle for the whole gmx process. Prefer a /tmp log_path.
    # Opening/appending on gcsfuse per line (or even frequent flush) can hang the kernel.
    log_fh = log_path.open("a", encoding="utf-8") if log_path is not None else None
    lines_since_flush = 0
    flush_every = 200  # /tmp is local; infrequent flush is fine

    def _append_log(text: str, *, flush: bool = False) -> None:
        nonlocal lines_since_flush
        if log_fh is None:
            return
        log_fh.write(text)
        lines_since_flush += 1
        if flush or lines_since_flush >= flush_every:
            log_fh.flush()
            lines_since_flush = 0

    try:
        if log_fh is not None:
            log_fh.write(f"$ {' '.join(cmd)}\n")
            log_fh.flush()

        if stream:
            proc = subprocess.Popen(
                cmd,
                cwd=cwd,
                stdin=subprocess.PIPE,
                stdout=subprocess.PIPE,
                stderr=subprocess.STDOUT,
                text=True,
            )
            if stdin is not None:
                proc.stdin.write(stdin)
                proc.stdin.close()
            stdout_chunks: list[str] = []
            last_milestone = 0
            progress_keys = (
                "Performance",
                "Finished mdrun",
                "Writing checkpoint",
            )
            for line in proc.stdout:
                stdout_chunks.append(line)
                _append_log(line)
                stripped = line.rstrip()
                if "Writing checkpoint" in stripped and MD_TMP_ROOT.exists():
                    sync_md_from_tmp(names=("md.cpt", "md.edr", "md.log"))
                if progress_interval and progress_total_steps:
                    match = _STEP_RE.search(stripped)
                    if match:
                        step = int(match.group(1))
                        milestone = (step // progress_interval) * progress_interval
                        if milestone > last_milestone:
                            last_milestone = milestone
                            msg = (
                                f"  MD progress: step {milestone:,}/"
                                f"{progress_total_steps:,} "
                                f"({100 * milestone / progress_total_steps:.0f}%)"
                            )
                            # Notebook stdout only — do not mirror every gmx line to the UI.
                            print(msg, flush=True)
                            _append_log(msg + "\n", flush=True)
                elif stripped and any(key in stripped for key in progress_keys):
                    # Sparse high-signal lines to the notebook even without step milestones.
                    print(stripped, flush=True)
            proc.wait()
            stdout = "".join(stdout_chunks)
            if proc.returncode != 0:
                err = stdout.strip()
                if err:
                    print(err[-8000:] if len(err) > 8000 else err, flush=True)
                raise RuntimeError(f"gmx failed ({proc.returncode}): {' '.join(args)}")
            return stdout

        result = subprocess.run(
            cmd,
            cwd=cwd,
            input=stdin,
            text=True,
            capture_output=True,
            check=False,
        )
        if result.stdout:
            _append_log(result.stdout, flush=True)
            if log_path is None:
                print(result.stdout[-4000:], flush=True)
        if result.returncode != 0:
            err = (result.stderr or result.stdout or "").strip()
            if err:
                print(err[-8000:] if len(err) > 8000 else err, flush=True)
            raise RuntimeError(f"gmx failed ({result.returncode}): {' '.join(args)}")
        return result.stdout
    finally:
        if log_fh is not None:
            log_fh.close()


def _stage(step: int, total: int, label: str, func):
    """Run one pipeline step with a numbered progress banner."""
    print(f"\n[{step}/{total}] {label} …", flush=True)
    t0 = time.perf_counter()
    result = func()
    elapsed = time.perf_counter() - t0
    print(f"    ✓ {label} ({elapsed:.0f}s)", flush=True)
    return result


MDPS = write_mdp_files()
print(
    "MDP files:",
    ", ".join(p.name for p in MDPS.values()),
    f"| EM {EM_STEPS:,}  NVT/NPT {NVT_STEPS:,}  production {TOTAL_STEPS:,} steps",
)
print(
    f"RE-RUN OK: production uses {MD_TMP_ROOT} (local disk). "
    f"Checkpoints every {CHECKPOINT_SAVE_STEPS:,} steps ({CHECKPOINT_SAVE_PS:.0f} ps). "
    "When md.cpt exists in workspace, RUN_MD resumes production only (no equilibration).",
    flush=True,
)


In [ ]:
def _read_pdb_atom_lines(path: Path) -> list[str]:
    """Return ATOM/HETATM records; repair OpenMM output missing newlines."""
    text = path.read_text()
    lines = [line for line in text.splitlines() if line.startswith(("ATOM", "HETATM"))]
    if len(lines) < text.count("ATOM  ") // 2:
        text = text.replace("ATOM  ", "\nATOM  ").lstrip("\n")
        lines = [line for line in text.splitlines() if line.startswith(("ATOM", "HETATM"))]
    return lines


def _pdb_residue_key(line: str) -> tuple[str, str, str]:
    """Return (chain_id, resseq, resname) from a fixed-width ATOM line."""
    resname = line[17:20].strip()
    chain = line[21]
    resseq = line[22:26].strip()
    if not resseq.isdigit():
        chain = line[20]
        resseq = line[21:25].strip()
    return chain, resseq, resname


def _prune_single_residue_chains(src: Path, dst: Path, *, min_residues: int = 2) -> Path:
    """Drop sequence segments too short for pdb2gmx (e.g. lone ARG)."""
    atom_lines = _read_pdb_atom_lines(src)
    residues: dict[tuple[str, str, str], list[str]] = defaultdict(list)
    residue_order: list[tuple[str, str, str]] = []
    for line in atom_lines:
        key = _pdb_residue_key(line)
        if key not in residues:
            residue_order.append(key)
        residues[key].append(line)

    segments: list[list[tuple[str, str, str]]] = []
    current: list[tuple[str, str, str]] = []
    current_chain: str | None = None
    prev_seq: int | None = None
    for key in residue_order:
        chain, resseq, _ = key
        seq = int(resseq)
        if current and (chain != current_chain or (prev_seq is not None and seq != prev_seq + 1)):
            segments.append(current)
            current = []
        current_chain = chain
        prev_seq = seq
        current.append(key)
    if current:
        segments.append(current)

    kept_keys: list[tuple[str, str, str]] = []
    pruned: list[str] = []
    for seg in segments:
        if len(seg) < min_residues:
            pruned.append(f"{seg[0][0]}:{seg[0][2]}{seg[0][1]}")
            continue
        kept_keys.extend(seg)

    serial = 1
    with dst.open("w") as handle:
        for idx, key in enumerate(kept_keys):
            if idx and _pdb_residue_key(residues[key][0])[0] != _pdb_residue_key(residues[kept_keys[idx - 1]][0])[0]:
                handle.write("TER\n")
            for line in residues[key]:
                handle.write(f"{line[:6]}{serial:5d}{line[11:]}\n")
                serial += 1
        handle.write("TER\nEND\n")

    if pruned:
        print(f"pruned single-residue segments: {', '.join(pruned)}")
    print(f"  → {serial - 1} atoms, {len(segments) - len(pruned)} chains")
    return dst


def fix_pdb_for_gromacs(src: Path, dst: Path) -> Path:
    """Add missing heavy atoms (docking structures are often incomplete)."""
    global PDBFile, PDBFixer
    if PDBFixer is None or PDBFile is None:
        print("openmm/pdbfixer not in this image — installing via mamba (one-time, ~2 min)...")
        print("Or rebuild permanently: cd tomo-plat/src && ./up-gromacs.sh --build")
        subprocess.run(
            ["mamba", "install", "-y", "-c", "conda-forge", "openmm", "pdbfixer"],
            check=True,
        )
        from openmm.app import PDBFile as _PDBFile
        from pdbfixer import PDBFixer as _PDBFixer

        PDBFile, PDBFixer = _PDBFile, _PDBFixer
    if PDBFixer is None or PDBFile is None:
        raise ImportError("pdbfixer/openmm required for fix_pdb_for_gromacs")
    fixer = PDBFixer(filename=str(src))
    fixer.findMissingResidues()
    fixer.findMissingAtoms()
    fixer.addMissingAtoms()
    tmp = dst.with_suffix(".pdbfixer.pdb")
    buf = io.StringIO()
    PDBFile.writeFile(fixer.topology, fixer.positions, buf, keepIds=True)
    tmp.write_text(buf.getvalue())
    _prune_single_residue_chains(tmp, dst)
    tmp.unlink(missing_ok=True)
    print(f"fixed: {dst}")
    return dst




def prepare_solvated_system(*, log_path: Path | None = None) -> Path:
    """pdb2gmx → box → solvate → neutralize. Returns path to solvated .gro."""
    total = 5

    def fix_and_topology():
        fixed_pdb = MD.root / "pocket_fixed.pdb"
        fix_pdb_for_gromacs(MD.pocket_pdb, fixed_pdb)
        run_gmx(
            [
                "pdb2gmx",
                "-f", fixed_pdb.name,
                "-o", MD.processed_gro.name,
                "-p", MD.topol_top.name,
                "-i", "posre.itp",
                "-ff", FORCEFIELD,
                "-water", WATER_MODEL,
                "-ignh",
            ],
            log_path=log_path,
        )

    _stage(1, total, "PDBFixer + pdb2gmx", fix_and_topology)

    boxed = MD.root / "pocket_box.gro"
    _stage(
        2, total, "Build dodecahedron box",
        lambda: run_gmx(
            ["editconf", "-f", MD.processed_gro.name, "-o", boxed.name, "-c", "-d", "1.0", "-bt", "dodecahedron"],
            log_path=log_path,
        ),
    )

    solv = MD.root / "pocket_solv.gro"
    _stage(
        3, total, "Solvate",
        lambda: run_gmx(
            ["solvate", "-cp", boxed.name, "-cs", "spc216.gro", "-o", solv.name, "-p", MD.topol_top.name],
            log_path=log_path,
        ),
    )

    tpr = MD.root / "ions.tpr"
    _stage(
        4, total, "grompp (ions)",
        lambda: run_gmx(
            ["grompp", "-f", MDPS["ions"].name, "-c", solv.name, "-p", MD.topol_top.name, "-o", tpr.name, "-maxwarn", "2"],
            log_path=log_path,
        ),
    )

    solv_ions = MD.root / "pocket_solv_ions.gro"
    _stage(
        5, total, "Add ions (0.15 M)",
        lambda: run_gmx(
            ["genion", "-s", tpr.name, "-o", solv_ions.name, "-p", MD.topol_top.name, "-neutral", "-conc", "0.15"],
            stdin="SOL\n",
            log_path=log_path,
        ),
    )
    print(f"    → solvated system: {solv_ions.name}", flush=True)
    return solv_ions


def production_mdrun_args(*, tmp_root: Path = MD_TMP_ROOT) -> list[str]:
    """gmx mdrun -v -deffnm md on local disk [-nb gpu …] [-cpi md.cpt].

    Run with cwd=MD_TMP_ROOT so md.xtc / md.cpt / md.edr / md.log stay off gcsfuse.
    """
    args = ["mdrun", "-v", "-deffnm", "md"]
    if USE_GPU_MDRUN:
        args.extend(MDRUN_GPU_FLAGS)
    if (tmp_root / "md.cpt").exists():
        args.extend(["-cpi", "md.cpt"])
    return args


def has_production_checkpoint(*, root: Path = MD.root) -> bool:
    """True when the workspace has a production checkpoint to resume from."""
    return (root / "md.cpt").is_file()


def run_production_mdrun(*, log_path: Path | None = None) -> None:
    """Production mdrun on /tmp; syncs checkpoint writes back to workspace."""
    progress_interval = max(1, TOTAL_STEPS // 100)
    run_gmx(
        production_mdrun_args(),
        cwd=MD_TMP_ROOT,
        log_path=log_path,
        progress_interval=progress_interval,
        progress_total_steps=TOTAL_STEPS,
    )
    sync_md_from_tmp()


def resume_production_from_checkpoint(*, log_path: Path | None = None) -> Path:
    """Copy latest checkpoint from workspace → /tmp and continue production only."""
    if not has_production_checkpoint():
        raise FileNotFoundError(f"No md.cpt under {MD.root} — run the full pipeline first.")

    sync_md_to_tmp()
    if not (MD_TMP_ROOT / "md.tpr").is_file():
        raise FileNotFoundError(
            f"md.cpt exists but md.tpr is missing under {MD.root}. "
            "Resume needs the matching .tpr from the same production run."
        )

    print(
        f"Resuming production from {MD.root / 'md.cpt'} on {MD_TMP_ROOT} "
        f"(checkpoint sync back to workspace every {CHECKPOINT_SAVE_PS:.0f} ps)",
        flush=True,
    )
    run_production_mdrun(log_path=log_path)

    xtc = MD.root / "md.xtc"
    if not xtc.exists():
        raise FileNotFoundError(f"Expected trajectory at {xtc} after resume")
    return xtc


print(
    f"RE-RUN OK: production_mdrun_args() runs under {MD_TMP_ROOT} with -cpi md.cpt when present. "
    "Confirm Idle time remaining > 0 before long mdrun.",
    flush=True,
)

def run_md_pipeline(
    solvated_gro: Path | None = None,
    *,
    log_path: Path | None = None,
    resume: bool | None = None,
) -> Path:
    """EM → NVT → NPT → production, or resume production only when md.cpt exists."""
    if resume is None:
        resume = has_production_checkpoint() and not FORCE_FROM_SCRATCH
    if resume:
        return resume_production_from_checkpoint(log_path=log_path)
    if solvated_gro is None:
        raise ValueError("solvated_gro is required for a from-scratch pipeline run")

    total = 8
    prod_ps = TOTAL_STEPS * MD_DT_PS
    nvt_ps = NVT_STEPS * MD_DT_PS
    npt_ps = NPT_STEPS * MD_DT_PS

    em_tpr = MD.root / "em.tpr"
    _stage(
        1, total, "grompp (energy minimization)",
        lambda: run_gmx(
            ["grompp", "-f", MDPS["em"].name, "-c", solvated_gro.name, "-p", MD.topol_top.name, "-o", em_tpr.name, "-maxwarn", "2"],
            log_path=log_path,
        ),
    )
    _stage(
        2, total, f"mdrun EM ({EM_STEPS:,} steps)",
        lambda: run_gmx(["mdrun", "-v", "-deffnm", "em"], log_path=log_path),
    )

    nvt_tpr = MD.root / "nvt.tpr"
    _stage(
        3, total, f"grompp (NVT, {nvt_ps:.1f} ps)",
        lambda: run_gmx(
            ["grompp", "-f", MDPS["nvt"].name, "-c", "em.gro", "-r", "em.gro", "-p", MD.topol_top.name, "-o", nvt_tpr.name, "-maxwarn", "2"],
            log_path=log_path,
        ),
    )
    _stage(
        4, total, f"mdrun NVT ({NVT_STEPS:,} steps)",
        lambda: run_gmx(["mdrun", "-v", "-deffnm", "nvt"], log_path=log_path),
    )

    npt_tpr = MD.root / "npt.tpr"
    _stage(
        5, total, f"grompp (NPT, {npt_ps:.1f} ps)",
        lambda: run_gmx(
            ["grompp", "-f", MDPS["npt"].name, "-c", "nvt.gro", "-r", "nvt.gro", "-t", "nvt.cpt", "-p", MD.topol_top.name, "-o", npt_tpr.name, "-maxwarn", "2"],
            log_path=log_path,
        ),
    )
    _stage(
        6, total, f"mdrun NPT ({NPT_STEPS:,} steps)",
        lambda: run_gmx(["mdrun", "-v", "-deffnm", "npt"], log_path=log_path),
    )

    def _production_grompp():
        sync_md_to_tmp()
        run_gmx(
            [
                "grompp",
                "-f", MDPS["md"].name,
                "-c", "npt.gro",
                "-t", "npt.cpt",
                "-p", MD.topol_top.name,
                "-o", "md.tpr",
                "-maxwarn", "2",
            ],
            cwd=MD_TMP_ROOT,
            log_path=log_path,
        )

    _stage(7, total, f"grompp (production, {prod_ps:.0f} ps) on /tmp", _production_grompp)

    _stage(
        8, total, f"mdrun production ({TOTAL_STEPS:,} steps) on /tmp",
        lambda: run_production_mdrun(log_path=log_path),
    )

    xtc = MD.root / "md.xtc"
    if not xtc.exists():
        raise FileNotFoundError(f"Expected trajectory at {xtc} (copied from {MD_TMP_ROOT})")
    return xtc


In [ ]:
if RUN_MD:
    run_log = open_md_run_log()
    print(f"Run log (live /tmp): {run_log}", flush=True)
    pipeline_t0 = time.perf_counter()
    resuming = has_production_checkpoint() and not FORCE_FROM_SCRATCH
    if resuming:
        print("=== Resuming GROMACS production from workspace checkpoint ===", flush=True)
    else:
        print("=== GROMACS pipeline start (from scratch) ===", flush=True)
    try:
        if resuming:
            traj_path = run_md_pipeline(log_path=run_log)
        else:
            solvated = prepare_solvated_system(log_path=run_log)
            solvated_elapsed = time.perf_counter() - pipeline_t0
            print(f"\n=== Solvation done ({solvated_elapsed:.0f}s) — starting MD ===", flush=True)
            traj_path = run_md_pipeline(solvated, log_path=run_log)
        total_elapsed = time.perf_counter() - pipeline_t0
        print(f"\n=== Pipeline complete ({total_elapsed:.0f}s) ===", flush=True)
        print("Production trajectory:", traj_path)
    finally:
        archived = finalize_md_run_log(run_log)
        print("Full run log:", archived or run_log)
else:
    print("RUN_MD=False — pocket/complex built; set RUN_MD=True to execute GROMACS.")


In [ ]:
# Process production MD (slow: loads md.xtc). FORCE_MD_REPROCESS is set in the first code cell.
MD_PROCESS_OK = False

xtc = MD.root / "md.xtc"
gro = MD.root / "npt.gro"
edr = MD.root / "md.edr"
final_pdb = MD.root / "md_final_protein.pdb"

if not xtc.exists() or not gro.exists():
    print("No md.xtc yet — run the GROMACS cell with RUN_MD=True.")
elif FORCE_MD_REPROCESS or RUN_MD or not final_pdb.exists():
    traj = load_md_trajectory(force_pbc=FORCE_MD_REPROCESS)
    print(traj)

    if edr.exists():
        ener = MD.root / "md_ener.xvg"
        try:
            run_gmx(["energy", "-f", "md.edr", "-o", "md_ener.xvg"], stdin="Potential\n")
        except RuntimeError:
            print("Skipping energy export (gmx energy failed).")
        else:
            print(f"Wrote {ener}")

    save_md_final_protein_pdb(traj, MD.pocket_pdb, final_pdb)
    print(f"Wrote {final_pdb}")
    MD_PROCESS_OK = True
else:
    print(
        "Skipping trajectory load — md_final_protein.pdb already exists. "
        "Run the display cell below, or set FORCE_MD_REPROCESS=True."
    )


In [ ]:
# Save manifest only when processing succeeded this run (run right after the cell above).
save_md_results_manifest(process_ok=globals().get("MD_PROCESS_OK", False))

In [ ]:
def make_nglview_view(pdb: str | Path):
    """Build an nglview widget for a PDB."""

    view = ng.show_file(str(pdb))
    view.clear_representations()
    view.add_representation("cartoon", color_scheme="chainindex")
    view.center()
    return view

def display_saved_md_structure() -> bool:
    """Show final-frame structure: full IGF1R context + pocket highlight + calx-β."""

    final_pdb, _, _ = _saved_md_result_paths()
    if not final_pdb.exists():
        print("No saved MD results — run the processing cell with RUN_MD=True first.")
        return False

    final_view = make_md_final_view(
                final_pdb,
                receptor_path=MD.receptor_clean,
                pocket_pdb=MD.pocket_pdb,
            )
    return final_view

def display_widget_snapshot(widget, poll_interval_s: float = 0.1, timeout_s: float = 10.0):
    """Render a widget image snapshot and display it inline."""

    image_widget = widget.render_image()
    waited = 0.0
    while not image_widget.value and waited < timeout_s:
        time.sleep(poll_interval_s)
        waited += poll_interval_s

    if not image_widget.value:
        raise TimeoutError("Widget snapshot timed out; try rerunning after viewer fully loads.")

    display(Image(data=image_widget.value, format="png"))
    return image_widget.value



In [ ]:
# Display from saved files — fast; works with RUN_MD=False if outputs exist.
final_view = display_saved_md_structure()
final_view

In [ ]:
# exportable image
display(
    Markdown(
        "**Production MD — final frame** "
        "(full IGF1R **gray**, pocket **red**, calx-β **blue**; "
        "receptor aligned to MD pocket Cαs)"
    )
)
_ = display_widget_snapshot(final_view)

In [ ]:
# Display energy/time
display_saved_md_energy()

In [ ]:
# Binding stability — compute ligand RMSD + receptor contact counts (loads md.xtc).
# FORCE_BINDING_REPROCESS is set in the first code cell.

metrics_path = md_binding_metrics_path()
if not metrics_path.exists() or FORCE_BINDING_REPROCESS:
    traj = load_md_trajectory()
    print(f"Computing binding metrics: {traj.n_frames} frames, {traj.time[-1]:.0f} ps")
    metrics = compute_binding_metrics(traj, MD.pocket_pdb)
    save_binding_metrics(metrics)
    display(metrics.describe().round(3))
else:
    print(f"Using cached {metrics_path} — set FORCE_BINDING_REPROCESS=True to recompute.")


In [ ]:
# Plot binding metrics and assess relative stability of calx-β in the IGF1R pocket.
_ = display_binding_analysis()

In [ ]:
! jupyter nbconvert --to html --no-input --embed-images igf1r3qqu_calxbeta_gromacs_md.ipynb